In [1]:
pip install pyspark==3.5.1

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys,os

print('Python',sys.executable)
print('Java',os.environ.get('JAVA_HOME'))
print('hadoop',os.environ.get('HADOOP_HOME'))

Python C:\Users\laboratorioesan\AppData\Local\anaconda3\python.exe
Java C:\Users\laboratorioesan\Desktop\jdk-17.0.12_windows-x64_bin\jdk-17.0.12
hadoop C:\Users\laboratorioesan\Desktop\winutils-master\winutils-master\hadoop-3.3.6


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("mllib-local").getOrCreate()

In [4]:
spark

In [5]:
os.getcwd()

'C:\\Users\\laboratorioesan'

In [6]:
ruta="C:\\Users\\laboratorioesan\\Desktop\\Datasets Repaso PC2-20260705\\"

In [7]:
# ── Capa Bronze ──────────────────────────────────────────────
b_buses  = spark.read.csv(ruta+'flota_buses.csv',  header=True, inferSchema=True)
b_cond   = spark.read.csv(ruta+'conductores.csv',  header=True, inferSchema=True)
b_tele11 = spark.read.option('multiLine', True).json(ruta+'telemetria_conduccion.json')
b_cond11 = spark.read.option('multiLine', True).json(ruta+'condiciones_ruta.json')

In [26]:
b_cond.printSchema()

root
 |-- id_unidad: string (nullable = true)
 |-- id_conductor: string (nullable = true)
 |-- fecha_actualizacion: timestamp (nullable = true)
 |-- antiguedad_conductor_anios: double (nullable = true)
 |-- categoria_licencia: string (nullable = true)
 |-- infracciones_historicas: double (nullable = true)



In [8]:
b_cond11.printSchema()

root
 |-- corredor_asignado: string (nullable = true)
 |-- id_unidad: string (nullable = true)
 |-- ruta: struct (nullable = true)
 |    |-- calidad_via_score: double (nullable = true)
 |    |-- indice_congestion: double (nullable = true)



In [9]:
from pyspark.sql.functions import (
    col, when, lit, count, avg, sum as spark_sum,
    datediff, to_date, row_number
)

from pyspark.sql import Window

In [10]:
# ── Capa Silver — Aplanado JSON ──────────────────────────────
df_tele11 = b_tele11.select(
    col('id_unidad'),
    col('comportamiento.frenadas_bruscas_dia').alias('frenadas_bruscas_dia'),
    col('comportamiento.excesos_velocidad_dia').alias('excesos_velocidad_dia')
)
df_cond11 = b_cond11.select(
    col('id_unidad'),
    col('ruta.indice_congestion').alias('indice_congestion'),
    col('ruta.calidad_via_score').alias('calidad_via_score')
)

In [25]:
df_tele11.describe().show()

+-------+---------+--------------------+---------------------+
|summary|id_unidad|frenadas_bruscas_dia|excesos_velocidad_dia|
+-------+---------+--------------------+---------------------+
|  count|     2000|                2000|                 2000|
|   mean|     NULL|               14.62|               9.6405|
| stddev|     NULL|   8.713355339973248|    5.724477668074716|
|    min|  BUS0001|                   0|                    0|
|    max|  BUS2000|                  29|                   19|
+-------+---------+--------------------+---------------------+



In [11]:
for c in ['frenadas_bruscas_dia', 'excesos_velocidad_dia']:
    q = df_tele11.approxQuantile(c, [0.5], 0.01)
    print(q)
    df_tele11 = df_tele11.fillna({c: q[0] if q else 0.0})

for c in ['indice_congestion', 'calidad_via_score']:
    q = df_cond11.approxQuantile(c, [0.5], 0.01)
    print(q)
    df_cond11 = df_cond11.fillna({c: q[0] if q else 0.0})

[14.0]
[10.0]
[0.5426]
[5.31]


In [12]:
w_cond  = Window.partitionBy('id_conductor').orderBy(col('fecha_actualizacion').desc())
df_cond = (b_cond
           .withColumn('rn', row_number().over(w_cond))
           .filter(col('rn') == 1).drop('rn'))

In [13]:
df_cond = df_cond.filter(col('antiguedad_conductor_anios') >= 0)

In [14]:
for c in ['infracciones_historicas', 'antiguedad_conductor_anios']:
    med = df_cond.approxQuantile(c, [0.5], 0.01)[0]
    df_cond = df_cond.fillna({c: med})

In [15]:
gold_flota = (b_buses
              .join(df_cond,   'id_unidad', 'inner')
              .join(df_tele11, 'id_unidad', 'inner')
              .join(df_cond11, 'id_unidad', 'inner'))

In [27]:
gold_flota.printSchema()

root
 |-- numero_unidad: string (nullable = true)
 |-- anno_fabricacion: integer (nullable = true)
 |-- capacidad_pasajeros: integer (nullable = true)
 |-- tipo_motor: string (nullable = true)
 |-- kilometraje_acumulado: double (nullable = false)
 |-- horas_servicio_dia: double (nullable = false)
 |-- fecha_actualizacion: timestamp (nullable = true)
 |-- antiguedad_conductor_anios: double (nullable = false)
 |-- categoria_licencia: string (nullable = true)
 |-- infracciones_historicas: double (nullable = false)
 |-- frenadas_bruscas_dia: long (nullable = true)
 |-- excesos_velocidad_dia: long (nullable = true)
 |-- indice_congestion: double (nullable = false)
 |-- calidad_via_score: double (nullable = false)
 |-- indice_riesgo_conductor: double (nullable = false)
 |-- desgaste_relativo: double (nullable = false)
 |-- exposicion_congestion: double (nullable = false)
 |-- siniestros_mes: double (nullable = false)



In [16]:
ANNO_ACT11 = 2026
gold_flota = gold_flota\
    .withColumn('indice_riesgo_conductor',
                (col('infracciones_historicas') * 5) + (col('frenadas_bruscas_dia') * 2))\
    .withColumn('desgaste_relativo',
                col('kilometraje_acumulado') / (lit(ANNO_ACT11) - col('anno_fabricacion') + 1))\
    .withColumn('exposicion_congestion',
                col('indice_congestion') * col('horas_servicio_dia'))\
    .withColumn('siniestros_mes',
                (col('indice_riesgo_conductor') * 0.04) +
                (col('desgaste_relativo') * 0.0001) +
                (col('excesos_velocidad_dia') * 0.8) +
                (col('calidad_via_score') * -0.5))

In [17]:
cols_to_drop = ['id_unidad', 'id_conductor']
gold_flota = gold_flota.drop(*[c for c in cols_to_drop if c in gold_flota.columns])

In [18]:
os.getcwd()

'C:\\Users\\laboratorioesan'

In [19]:
from ClaseFeatureSelectorBDA import FeatureSelectorBDA

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lit, count, avg, sum as spark_sum,
    datediff, to_date, row_number
)
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)

# Clasificación
from pyspark.ml.classification import (
    DecisionTreeClassifier, RandomForestClassifier,
    LogisticRegression, GBTClassifier
)
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator
)

# Regresión
from pyspark.ml.regression import (
    LinearRegression, RandomForestRegressor,
    GBTRegressor, DecisionTreeRegressor, GeneralizedLinearRegression
)
from pyspark.ml.evaluation import RegressionEvaluator

In [20]:
# ─────────────────────────────────────────────────────────────
# FUNCIONES AUXILIARES REUTILIZABLES
# ─────────────────────────────────────────────────────────────

def evaluar_clasificacion(predictions, label_col):
    """Imprime Accuracy, F1, Precision y Recall para un modelo de clasificación."""
    metrics = ['accuracy', 'f1', 'weightedPrecision', 'weightedRecall']
    nombres = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
    evaluator = MulticlassClassificationEvaluator(
        labelCol=label_col, predictionCol='prediction'
    )
    resultados = {}
    for m, n in zip(metrics, nombres):
        resultados[n] = round(evaluator.setMetricName(m).evaluate(predictions), 4)
    print(resultados)


reg_eval = RegressionEvaluator(predictionCol='prediction')

def evaluar_regresion(train_df, test_df, label_col, modelos_dict):
    """Entrena y evalúa múltiples modelos de regresión. Imprime RMSE, MAE y R2."""
    print(f"\n{'Modelo':30s} | RMSE      | MAE       | R2")
    print('-' * 65)
    for name, est in modelos_dict.items():
        est.setLabelCol(label_col).setFeaturesCol('features_scaled')
        model = est.fit(train_df)
        pred  = model.transform(test_df)
        rmse = reg_eval.setLabelCol(label_col).setMetricName('rmse').evaluate(pred)
        mae  = reg_eval.setLabelCol(label_col).setMetricName('mae').evaluate(pred)
        r2   = reg_eval.setLabelCol(label_col).setMetricName('r2').evaluate(pred)
        print(f"{name:30s} | {rmse:.3f}   | {mae:.3f}   | {r2:.3f}")


def impute_all_numerics(df, label_col):
    from pyspark.sql.types import (DoubleType, FloatType, IntegerType,
                                   LongType, ShortType)
    numeric_types = (DoubleType, FloatType, IntegerType, LongType, ShortType)
    num_cols = [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, numeric_types) and f.name != label_col
    ]
    fill_map = {}
    for c in num_cols:
        try:
            med = df.approxQuantile(c, [0.5], 0.01)
            if med:
                fill_map[c] = med[0]
        except Exception:
            fill_map[c] = 0.0
    if fill_map:
        df = df.fillna(fill_map)
    return df


def build_pipeline_clasificacion(cat_cols, num_cols):
    stages = []
    for c in cat_cols:
        stages.append(StringIndexer(inputCol=c, outputCol=f'{c}_idx', handleInvalid='keep'))
        stages.append(OneHotEncoder(inputCol=f'{c}_idx', outputCol=f'{c}_ohe'))
    final_inputs = [f'{c}_ohe' for c in cat_cols] + num_cols
    stages.append(VectorAssembler(
        inputCols=final_inputs,
        outputCol='features',
        handleInvalid='keep'          # ← clave: tolera nulos residuales
    ))
    stages.append(StandardScaler(
        inputCol='features', outputCol='features_scaled',
        withMean=True, withStd=True
    ))
    return Pipeline(stages=stages)



In [21]:
SEED = 700

In [22]:
etiqueta11 = 'siniestros_mes'
gold_flota = impute_all_numerics(gold_flota, etiqueta11)
gold_flota = gold_flota.fillna(0.0)

fs11 = FeatureSelectorBDA(gold_flota, etiqueta11)
cat11, num11 = fs11.division_columnas()
num_filt11, _, _ = fs11.get_multicolinealidad()
num_sel11 = fs11.get_cols_selectednum(num_filt11)
cat_sel11 = fs11.get_cols_selected(featureTypeCat=True, cols=cat11) if cat11 else []

df_sel11 = gold_flota.select(etiqueta11, *cat_sel11, *num_sel11)
train11, test11 = df_sel11.randomSplit([0.8, 0.2], seed=SEED)

pipe11     = build_pipeline_clasificacion(cat_sel11, num_sel11)
pipe_fit11 = pipe11.fit(train11)
train_p11  = pipe_fit11.transform(train11)
test_p11   = pipe_fit11.transform(test11)

In [23]:
modelos11 = {
    'LinearRegression':            LinearRegression(),
    'GeneralizedLinearRegression': GeneralizedLinearRegression(family='gaussian', link='identity'),
    'RandomForestRegressor':       RandomForestRegressor(numTrees=100, seed=SEED)
}
evaluar_regresion(train_p11, test_p11, etiqueta11, modelos11)


Modelo                         | RMSE      | MAE       | R2
-----------------------------------------------------------------
LinearRegression               | 0.000   | 0.000   | 1.000
GeneralizedLinearRegression    | 0.000   | 0.000   | 1.000
RandomForestRegressor          | 1.143   | 0.922   | 0.948
